In [2]:
import pandas as pd 
df = pd.read_csv('Data/drug_review_train.csv')

df.head()

,Unnamed: 0,patient_id,drugName,condition,review,rating,date,usefulCount,review_length
0,0,89879,Cyclosporine,keratoconjunctivitis sicca,"""i have used restasis for about a year now and...",2.0,"April 20, 2013",69,147
1,1,143975,Etonogestrel,birth control,"""my experience has been somewhat mixed. i have...",7.0,"August 7, 2016",4,136
2,2,106473,Implanon,birth control,"""this is my second implanon would not recommen...",1.0,"May 11, 2016",6,140
3,3,184526,Hydroxyzine,anxiety,"""i recommend taking as prescribed, and the bot...",10.0,"March 19, 2012",124,104
4,4,91587,Dalfampridine,multiple sclerosis,"""i have been on ampyra for 5 days and have bee...",9.0,"August 1, 2010",101,74


In [2]:
df.drop(columns=['Unnamed: 0','date', 'usefulCount', 'review_length'], inplace=True)

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
import string 

X_train = df['review']
y_train = df['rating']

import ssl
import nltk

# Fix SSL issue
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Now download
nltk.download('punkt')
nltk.download('stopwords')


def preprocess(text):
    text = text.lower()  # Lowercase
    tokens = word_tokenize(text)  # Tokenize
    tokens = [word for word in tokens if word not in stop_words and word not in punctuation]  # Remove stopwords and punctuation
    return ' '.join(tokens)  # Rejoin tokens

# Apply preprocessing
cleaned_reviews = [preprocess(isinstancee) for isinstancee in X_train]

# TF-IDF vectorization
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(cleaned_reviews)


[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1000)>
[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1000)>


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/Users/youssefbenmansour/nltk_data'
    - '/Library/Frameworks/Python.framework/Versions/3.12/nltk_data'
    - '/Library/Frameworks/Python.framework/Versions/3.12/share/nltk_data'
    - '/Library/Frameworks/Python.framework/Versions/3.12/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [29]:
import pandas as pd
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


# Load data
df = pd.read_csv("/Users/youssefbenmansour/Downloads/HCML-NLP-Project-main/Data/drug_review_train.csv")  # replace with your actual path
df.drop(columns=['Unnamed: 0','date', 'usefulCount', 'review_length'], inplace=True)
# Drop rows with missing reviews or ratings
df = df.dropna(subset=['review', 'rating'])

# Basic preprocessing
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

import re
import string



from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def preprocess_with_stopwords(text: str) -> str:
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]
    
    return ' '.join(tokens)

# Apply preprocessing
df['cleaned_review'] = df['review'].astype(str).apply(preprocess_with_stopwords)
# TF-IDF vectorization

vectorizer = TfidfVectorizer(max_features=5000)  # limit to 5000 features
X = vectorizer.fit_transform(df['cleaned_review'])

# Regression target
y = df['rating'].astype(float)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse:.2f}")


Mean Squared Error: 5.83


In [30]:
X_train_full, X_test_full, y_train, y_test_full = train_test_split(
    df[['review', 'cleaned_review']], df['rating'], test_size=0.2, random_state=42
)

# Create DataFrame with comparison
results_df = pd.DataFrame({
    'review': X_test_full['review'].values,
    'true_rating': y_test.values,
    'predicted_rating': y_pred
})

# Optional: Round predicted ratings to 2 decimals
results_df['predicted_rating'] = results_df['predicted_rating'].round(0)

mse = mean_squared_error(y_test, results_df['predicted_rating'])
print(f"Mean Squared Error: {mse:.2f}")
print(results_df.shape)
# Display first few
results_df.head(10)

Mean Squared Error: 5.92
(22163, 3)


,review,true_rating,predicted_rating
0,"""i've been on aviane for a little over a year....",9.0,9.0
1,"""i was diagnosed with obsessive compulsive dis...",9.0,6.0
2,"""i was put on humira on june 10, 2015 for seve...",1.0,3.0
3,"""i tried andro-gel for about 6-8 months. i did...",4.0,8.0
4,"""i'm calling this the 'closer to perfection dr...",10.0,7.0
5,"""this is my second colonoscopy. first one was ...",10.0,7.0
6,"""wow, i wish i had these years ago. i have ma...",10.0,9.0
7,"""due to my highly irregular and heavy periods ...",8.0,8.0
8,"""have taken this medicine intermittently 2.5 t...",6.0,9.0
9,"""taking effexor in the beginning was hard to g...",7.0,7.0
